# Chapter 15: Forecasting. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/15-forecasting.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/15-forecasting.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/15-forecasting.ipynb)

In [ ]:
# Packages that the exercises of this notebook use; the ones that are missing are installed
pkgs <- c("bsts", "quantregForest", "ranger")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org")

# The exercises read files from data/: download the datasets of the book once
if (!dir.exists("data")) {
  options(timeout = max(3600, getOption("timeout")))
  download.file("https://github.com/VadimSokolov/bayes_ai_deep_learning/releases/latest/download/data.zip", "data.zip", mode = "wb")
  unzip("data.zip")
  unlink("data.zip")
}

## Exercise 15.6: Structural Time Series with bsts

Use the `AirPassengers` dataset (monthly airline passengers, 1949–1960) to fit structural time series models.

a) Plot the time series. Does it appear to have a trend? Seasonality? Is the variance constant?

b) Apply a log transformation and fit a BSTS model with a local linear trend and monthly seasonality ($S=12$). Plot the fitted components.

c) Hold out the last 12 months. Fit the model on the training data and produce 12-month-ahead forecasts. Plot the forecasts with prediction intervals.

d) Compare the one-step-ahead prediction errors for models with and without the seasonal component using mean absolute one-step-ahead error (MAE).

## Exercise 15.7: Kalman Filter from Scratch

Implement a simple Kalman filter for the random walk plus noise model: $x_{t+1} = x_t + w_t$, $y_t = x_t + v_t$.

a) Generate 100 observations from this model with $\sigma_w = 1$ and $\sigma_v = 2$. Plot the true state and noisy observations.

b) Implement the Kalman filter: prediction step, update step, and Kalman gain. Filter the observations and plot the filtered state alongside the true state.

c) Plot the Kalman gain $K_t$ over time. Does it converge? To what value? (Report the iteration at which |K_t - K_{t-1}| first falls below 1e-6.)

d) Repeat with $\sigma_v = 0.1$ (precise observations). How does the filtered estimate change?

## Exercise 15.8: Dynamic Logistic State-Space Model for a Binary Series

The recession-modeling example in this chapter fits a dynamic logistic regression with a random-walk level, $\mathrm{logit}(p_t) = \mu_t + \beta^\top x_t$, $\mu_{t+1} = \mu_t + \eta_t$, using `bsts`. Here you simulate a simplified version (no regressors) and filter it yourself: the Bernoulli observation equation means the closed-form Kalman recursions of the previous exercise no longer apply, so a simulation-based filter is needed instead.

a) Simulate a latent log-odds $\theta_t$ following a Gaussian random walk for $T=150$ periods: $\theta_1 = -1$ and $\theta_{t+1} = \theta_t + \eta_t$ with $\eta_t \sim N(0, \sigma_w^2)$, $\sigma_w = 0.35$. Set $p_t = \mathrm{logit}^{-1}(\theta_t)$ and draw $y_t \sim \text{Bernoulli}(p_t)$, where $y_t=1$ might represent "recession this month." Plot $p_t$ together with the observed $y_t$.

b) The observation equation is Bernoulli, not Gaussian, so the linear-Gaussian Kalman filter of the previous exercise no longer gives a closed-form posterior for $\theta_t$. Implement a bootstrap particle filter: represent the filtering distribution of $\theta_t$ with $N$ weighted particles $\theta_t^{(i)}$, propagate each particle through the random-walk transition, reweight it by the Bernoulli likelihood of $y_t$, and resample. Report the filtered probability $\hat p_t = \mathrm{E}\left(p_t \mid y_{1:t}\right) \approx \sum_i w_t^{(i)}\,\mathrm{logit}^{-1}(\theta_t^{(i)})$.

c) Compute the mean absolute tracking error $\frac{1}{T}\sum_t |\hat p_t - p_t|$ and compare it to a naive baseline that always predicts the unconditional mean $\bar y$. Report both numbers.

d) Refit the filter assuming a process noise $\sigma_w = 0.05$, understating the true value. Does the filtered probability lag behind the true swings in $p_t$? Report the new tracking error and explain the result using the Kalman-gain intuition from the previous exercise.

## Exercise 15.9: Regime-Switching Volatility for S&P 500 Returns via Baum-Welch and Viterbi

The chapter's HMM and regime-switching sections describe a hidden Markov chain $x_t$ that switches the parameters of an observation process, $y_t = \mu(x_t) + \sigma(x_t)\varepsilon_t$, and note that FFBS-type recursions can be adapted to estimate such models. Here you fit a two-regime Gaussian HMM to real financial data by implementing scaled Baum-Welch (EM) and Viterbi decoding from scratch (Baum et al. 1970; Viterbi 1967).

a) Load `data/GSPC.csv` (daily S&P 500 prices, already used in this chapter) and compute daily log returns $r_t = 100(\log p_t - \log p_{t-1})$.

b) Implement Baum-Welch for a two-state Gaussian HMM, $r_t \mid x_t = j \sim N(\mu_j, \sigma_j^2)$, with unknown transition matrix $P$, unknown $(\mu_j, \sigma_j)$, and unknown initial distribution. Use scaled forward-backward recursions (dividing by $c_t = p(y_t \mid y_{1:t-1})$ at each step, as in the chapter's filtering recursions) to avoid underflow, iterating the E-step and M-step until the log-likelihood $\sum_t \log c_t$ changes by less than $10^{-6}$. Initialize by splitting returns into a near-median 70 percent group (low-volatility) and an extreme 30 percent group (high-volatility).

c) Label the state with the smaller $\sigma_j$ as low-volatility and the other as high-volatility. Report the fitted transition matrix, the state-dependent mean and standard deviation of daily returns, and the expected regime duration $1/(1 - P_{jj})$ in trading days for each regime.

d) Decode the Viterbi path and plot the filtered probability $\mathrm{P}\left(x_t = \text{high-vol} \mid y_{1:t}\right)$ over time. Report the mean filtered high-volatility probability during the COVID crash window (Feb 15 to Apr 15, 2020) versus calendar year 2017, and comment on whether the model isolates the crash as a distinct regime.

## Exercise 15.10: Quantile Regression Forests for Predictive Intervals

The chapter's discussion of quantile regression forests shows how a random forest can estimate the full conditional distribution $\hat p(y \mid x)$, not just the conditional mean $\mathrm{E}\left(Y \mid X\right)$, by keeping track of which training values land in each leaf. Here you fit one on simulated heteroskedastic data and check whether its predictive intervals are actually calibrated.

a) Simulate $n=800$ points with $x \sim \text{Uniform}(0,10)$ and $y = 3 + 2\sin(x) + \varepsilon$, where $\varepsilon \sim N(0, \sigma(x)^2)$ and $\sigma(x) = 0.4 + 0.35x$ grows with $x$. Split into a training set (first 600 points) and a test set (last 200 points).

b) Check whether the `ranger` package is installed with `requireNamespace()`. If so, fit a quantile regression forest with `quantreg = TRUE` and predict the 10th, 50th, and 90th percentiles of $y \mid x$ on the test set using `predict(..., type = "quantiles")`. If only `quantregForest` is available, use it instead. If neither package is installed, fall back to ordinary least squares plus the empirical quantiles of the training residuals, and note that this fallback only calibrates the interval on average, not conditionally on $x$.

c) Using the 10th and 90th percentile predictions as an 80% interval, compute its empirical coverage: the fraction of test points whose $y$ falls between the two predicted quantiles. How close is it to the nominal 0.80?

d) Now build a naive constant-width interval: add the 10th and 90th percentile of the training residuals to the median prediction, so every test point gets the same interval width. Compute the coverage of both intervals separately for $x<3$ and for $x>7$. Which method stays close to 80% coverage in both regions, and which one over-covers where the noise is small and under-covers where the noise is large?